In [14]:
import json
import pandas as pd
import numpy as np
import baybe
from baybe.insights import SHAPInsight
import os
import matplotlib.pyplot as plt

from pathlib import Path
import re
import pickle

import InitializeCampaign as ic
import InitializeCampaign as ic
import shap
from baybe.insights import SHAPInsight
from shap.explainers import *
import shapexplainers as se


In [15]:
#folder_path
HEAD_PATH = Path(r"G:\Mitarbeiter_208\rona\Projects\Active-Learning - Bipolar HiPIMS deposition rates\Bipolar Datasets - Al and Ti - low and high PW\Clean Datasets Used for Publication")

df_campaign = pd.DataFrame()

folder_list =[Path(r"G:\Mitarbeiter_208\rona\Projects\Active-Learning - Bipolar HiPIMS deposition rates\Bipolar Datasets - Al and Ti - low and high PW\Clean Datasets Used for Publication\Al - 120 W  - short PW")]
# for folder in os.listdir(HEAD_PATH):
for folder in folder_list:
    FOLDER_PATH = HEAD_PATH / folder
    #oscci and sys logfile paths
    CAMPAIGN_PATH = Path(FOLDER_PATH / 'Campaign.json')
    OSCCI_PATH = FOLDER_PATH / 'Logfile - Oscilloscope'
    LOGFILE_PATH = FOLDER_PATH / 'Logfiles'

    #Load the campaign file for easier compiling of data for later analysis
    CAMPAIGN_PATH = CAMPAIGN_PATH.resolve()
    json_str = open(CAMPAIGN_PATH).read()
    jay = json.loads(json_str)
    campaign = baybe.Campaign.from_json(jay)

    #import calibrations
    df = pd.read_csv(FOLDER_PATH / 'calibration.txt')
    us_window = int(df['us-window']) #us range of ossci sampling
    dep_rate_calibration = float(df[' dep-rate calibration']) #calibration of material system

    #print params and lower/upper bounds
    params = [p.name for p in campaign.parameters]
    print(f'The parameter names are: {params}')
    #Visually, if the minimum peak-current are not close to 0, trigger settings are off
    Ipk_list = []
    id=0
    for file in os.listdir(OSCCI_PATH):
        df_pw = campaign.measurements
        #import relevant data from Oscci LogFiles
        df = pd.read_json(str(OSCCI_PATH) + '\\' + file)

        corr_time = (us_window/len(df)) #in units of us/point, correct time for window in LabView
        #Create adjusted time
        df['Time'] = np.linspace(0,int(len(df)*corr_time),len(df)) #modify x for the range specified in sofware and the #of sampled points (almost always 10,000)
        
        #Visualize trigger to remove oscillation from peak-current determination
        trigger = 900*corr_time #est. where trigger is
        trigger_exclude = 250*corr_time #est. of # of samples to exclude post-trigger from Ipk determination 
        cutoff_trigger = trigger + trigger_exclude
        cutoff_pulse = cutoff_trigger + df_pw['PW (us)'][id] #add pulse-width to determination zone, avoids subseuqnet oscillations getting logged as Ipk
        
        #use defined triggers for Ipk determination
        df_mask = df.loc[((df['Time']> cutoff_trigger) & (df['Time'] < cutoff_pulse))] #finds peak-current within shaded green area, currently based on sample count, should be adjusted to pulse-width in the future
        
        Ipk_list.append(np.max(df_mask[2])/(45.58)) #units [A/cm2] - peak current density correction for 3" target
        #append power as well
        #Power_list.append(float(power/45.58))
        id +=1

    #CONSTRUCT DF FOR CAMPAIGN
    df_campaign = campaign.measurements #store campaign measurements into df

    df_campaign['Ipk (A)'] = Ipk_list
    
    #calibrate dep-rate
    df_campaign['y1'] = df_campaign['y1']*1000*dep_rate_calibration #convert from kA/s to A/s and correct for density of Al or Ti based on calibration file

    #define params, remove params included in campaign not wanted in SHAP, add those needed
    params = ['Ipk (A)','PW (us)','PRR (Hz)','pos. Delay (us)', 'pos. PW (us)', 'pos. Setpoint (V)']

    #handle duty cycle cases
    if ('PRR (Hz)' not in df_campaign.columns):
        df_campaign['PRR (Hz)'] = (df_campaign['Duty Cycle (ratio)'] / (df_campaign['PW (us)'] *1E-6))

    #define bounds
    lower_bounds,upper_bounds = [],[]
    for p in params:
        lower_bounds.append(np.min(df_campaign[p]))
        upper_bounds.append(np.max(df_campaign[p]))

    #Example of making a new campaign
    campaign = ic.init_campaign(lower_bounds, upper_bounds,params)

    #modify df_campaign to have only params of interest
    df_clean = pd.DataFrame()
    for p in params:
        df_clean[p] = df_campaign[p]
    df_clean['y1']=df_campaign['y1']
    campaign.add_measurements(df_clean)

    #define SHAP explanation to use
    insight = se.insight_partition(campaign)
    explanation = insight.explain()

    #pickel the object
    with open(folder + '.pkl', "wb") as f:
        pickle.dump(explanation, f)



C:\Users\rona\AppData\Local\Temp\ipykernel_43924\3603833560.py:23: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  us_window = int(df['us-window']) #us range of ossci sampling
C:\Users\rona\AppData\Local\Temp\ipykernel_43924\3603833560.py:24: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  dep_rate_calibration = float(df[' dep-rate calibration']) #calibration of material system


The parameter names are: ['PRR (Hz)', 'PW (us)', 'pos. Delay (us)', 'pos. PW (us)', 'pos. Setpoint (V)']


PartitionExplainer explainer: 602it [01:59,  4.80it/s]                         


TypeError: unsupported operand type(s) for +: 'WindowsPath' and 'str'

In [16]:
#pickel the object
with open(folder / '.pkl', "wb") as f:
    pickle.dump(explanation, f)